# SGLang + Docker 容器化部署 — 一键打包教程

**定位：演示如何使用 Docker 容器化部署 SGLang 推理服务，涵盖官方镜像、自定义 Dockerfile 和 Docker Compose 三种方式。**

> **默认启动命令：**
> ```bash
> python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
> ```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台。*

---

## 硬件与环境要求

| 项目 | 最低要求 |
|------|----------|
| **GPU** | NVIDIA GPU（算力 ≥ 7.0） |
| **显存（VRAM）** | ≥ 6 GB（Qwen2.5-0.5B） |
| **驱动** | NVIDIA Driver ≥ 525 |
| **CUDA** | ≥ 12.1 |
| **Python** | 3.9 – 3.12 |
| **操作系统** | Linux x86_64 最佳；Windows 建议使用 WSL2 |
| **Docker** | Docker Engine ≥ 20.10 + NVIDIA Container Toolkit |

---

## 依赖安装

```bash
# 安装 SGLang 全部组件及 OpenAI SDK
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

```bash
# 安装 Docker（如尚未安装）
# Ubuntu/Debian:
curl -fsSL https://get.docker.com | sh

# 安装 NVIDIA Container Toolkit
distribution=$(. /etc/os-release;echo $ID$VERSION_ID)
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
sudo apt-get update && sudo apt-get install -y nvidia-container-toolkit
sudo nvidia-ctk runtime configure --runtime=docker
sudo systemctl restart docker
```

---

## 使用指南

1. 按顺序执行每个代码单元格
2. 由于 Docker 操作主要在终端执行，本 Notebook 侧重于 **生成命令和配置文件**
3. 将生成的命令复制到终端中执行即可部署
4. 容器启动后可在本 Notebook 中验证服务状态

---

In [ ]:
# === 环境检查 ===
import subprocess  # 导入子进程模块，用于执行系统命令
import sys  # 导入系统模块，获取 Python 版本信息

print("=" * 50)  # 打印分隔线
print("环境检查")  # 打印标题
print("=" * 50)  # 打印分隔线

# 检查 Python 版本
print(f"Python 版本: {sys.version}")  # 输出当前 Python 版本

# 检查 GPU 是否可用
try:
    result = subprocess.run(  # 执行 nvidia-smi 命令查询 GPU 信息
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],  # 查询 GPU 名称和显存
        capture_output=True, text=True, timeout=10  # 捕获输出，设置超时
    )
    if result.returncode == 0:  # 如果命令执行成功
        print(f"GPU 信息: {result.stdout.strip()}")  # 输出 GPU 信息
    else:
        print("警告: nvidia-smi 执行失败")  # 输出警告
except FileNotFoundError:  # 如果找不到 nvidia-smi
    print("警告: 未找到 nvidia-smi，请确认已安装 NVIDIA 驱动")  # 输出警告

# 检查 sglang 是否已安装
try:
    import sglang  # 尝试导入 sglang
    print(f"SGLang 版本: {sglang.__version__}")  # 输出 sglang 版本
except ImportError:  # 如果未安装
    print("SGLang 未安装，请运行下方安装单元格")  # 提示安装

# 检查 openai 包
try:
    import openai  # 尝试导入 openai
    print(f"OpenAI SDK 版本: {openai.__version__}")  # 输出 openai 版本
except ImportError:  # 如果未安装
    print("OpenAI SDK 未安装")  # 提示安装

print("=" * 50)  # 打印分隔线

In [ ]:
# === 可选：安装依赖（已安装可跳过） ===
# !pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"  # 取消注释以安装依赖

## 为什么用 Docker 部署？

在生产环境中，使用 Docker 部署 SGLang 有以下核心优势：

1. **环境一致性**：将 Python、CUDA、SGLang 及其依赖全部打包进镜像，避免「我本地能跑，服务器不行」
2. **可复现性**：镜像版本固定，任何机器拉取后都能得到相同的运行环境
3. **隔离性**：容器之间互不干扰，一台服务器可同时部署多个模型实例
4. **快速扩缩容**：配合 Kubernetes 等编排工具可实现弹性伸缩
5. **运维标准化**：日志、监控、重启策略统一管理

本 Notebook 介绍三种容器化部署方式：
- **方式一**：直接使用 SGLang 官方 Docker 镜像（最快）
- **方式二**：自定义 Dockerfile（灵活定制）
- **方式三**：Docker Compose 编排（适合多服务场景）

In [ ]:
# === Docker 环境检查 ===
import subprocess  # 导入子进程模块

print("=" * 50)  # 打印分隔线
print("Docker 环境检查")  # 打印标题
print("=" * 50)  # 打印分隔线

# 检查 Docker 是否可用
try:
    docker_result = subprocess.run(  # 执行 docker --version 命令
        ["docker", "--version"],  # 查询 Docker 版本
        capture_output=True, text=True, timeout=10  # 捕获输出，设置超时
    )
    if docker_result.returncode == 0:  # 如果命令执行成功
        print(f"✓ Docker 已安装: {docker_result.stdout.strip()}")  # 输出 Docker 版本
    else:
        print("✗ Docker 未正确安装或未运行")  # 提示安装
except FileNotFoundError:  # 如果找不到 docker 命令
    print("✗ 未找到 docker 命令，请先安装 Docker")  # 提示安装

# 检查 NVIDIA Container Toolkit（通过运行 GPU 容器测试）
print("\n正在检查 NVIDIA Container Toolkit（可能需要几秒）...")  # 打印检查提示
try:
    gpu_test = subprocess.run(  # 执行 GPU 容器测试命令
        ["docker", "run", "--rm", "--gpus", "all",  # 使用 --gpus all 参数
         "nvidia/cuda:12.4.0-base-ubuntu22.04", "nvidia-smi"],  # 在 CUDA 容器中运行 nvidia-smi
        capture_output=True, text=True, timeout=120  # 捕获输出，超时 120 秒（首次需拉取镜像）
    )
    if gpu_test.returncode == 0:  # 如果命令执行成功
        print("✓ NVIDIA Container Toolkit 工作正常")  # 确认 GPU 容器支持
        gpu_lines = gpu_test.stdout.strip().split("\n")  # 按行分割输出
        for line in gpu_lines[:5]:  # 只显示前 5 行
            print(f"  {line}")  # 输出 nvidia-smi 信息
    else:
        print("✗ GPU 容器测试失败")  # 提示失败
        print(f"错误信息: {gpu_test.stderr.strip()}")  # 输出错误详情
except FileNotFoundError:  # 如果找不到 docker
    print("✗ 无法运行 Docker 命令")  # 提示错误
except subprocess.TimeoutExpired:  # 如果超时
    print("✗ GPU 容器测试超时，可能是首次拉取镜像耗时较长")  # 提示超时

print("=" * 50)  # 打印分隔线

## 方式一：使用官方 Docker 镜像

SGLang 项目提供了预构建的 Docker 镜像 `lmsysorg/sglang:latest`，内含全部依赖，是最快的部署方式。

**关键参数说明：**
- `--gpus all`：将宿主机所有 GPU 传入容器
- `-p 30000:30000`：将容器端口映射到宿主机
- `-v ~/.cache/huggingface:/root/.cache/huggingface`：挂载模型缓存目录，避免重复下载
- `--host 0.0.0.0`：容器内监听所有网卡（必须，否则外部无法访问）

In [ ]:
# === 方式一：生成官方镜像运行命令 ===

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # 定义使用的模型 ID
PORT = 30000  # 定义服务端口

# 构建 docker run 命令
docker_run_cmd = f"""docker run --gpus all \\  # 使用所有 GPU
  -p {PORT}:{PORT} \\  # 映射端口到宿主机
  -v ~/.cache/huggingface:/root/.cache/huggingface \\  # 挂载模型缓存目录
  --shm-size=16g \\  # 设置共享内存为 16GB（大模型推理需要）
  lmsysorg/sglang:latest \\  # 使用 SGLang 官方镜像
  python -m sglang.launch_server \\  # 启动 SGLang 推理服务
  --model-path {MODEL_ID} \\  # 指定模型路径
  --host 0.0.0.0 \\  # 监听所有网卡
  --port {PORT}"""  # 指定服务端口

print("=" * 60)  # 打印分隔线
print("方式一：官方镜像运行命令")  # 打印标题
print("=" * 60)  # 打印分隔线
print()  # 打印空行
print("请在终端中执行以下命令：")  # 提示用户
print()  # 打印空行

# 打印格式化的命令（去掉注释）
clean_cmd = f"""docker run --gpus all \\
  -p {PORT}:{PORT} \\
  -v ~/.cache/huggingface:/root/.cache/huggingface \\
  --shm-size=16g \\
  lmsysorg/sglang:latest \\
  python -m sglang.launch_server \\
  --model-path {MODEL_ID} \\
  --host 0.0.0.0 \\
  --port {PORT}"""  # 生成可直接复制的命令

print(clean_cmd)  # 输出完整命令
print()  # 打印空行
print("各参数含义：")  # 打印参数说明标题
print("  --gpus all          → 将宿主机所有 GPU 传入容器")  # 说明 GPU 参数
print(f"  -p {PORT}:{PORT}    → 将容器端口映射到宿主机")  # 说明端口映射
print("  -v ...              → 挂载 HuggingFace 缓存，避免重复下载模型")  # 说明卷挂载
print("  --shm-size=16g      → 增大共享内存，防止推理时 OOM")  # 说明共享内存
print("  --host 0.0.0.0      → 容器内监听所有接口，外部才能访问")  # 说明监听地址

## 方式二：自定义 Dockerfile

如果需要对镜像进行定制（如安装额外的 Python 包、添加配置文件等），可以编写自定义 Dockerfile。

**适用场景：**
- 需要在镜像中预装额外依赖
- 需要自定义入口脚本
- 需要固定特定版本的 SGLang

In [ ]:
# === 方式二：生成自定义 Dockerfile ===
import os  # 导入操作系统模块
import tempfile  # 导入临时文件模块

# 定义 Dockerfile 内容
dockerfile_content = """# 基础镜像：NVIDIA CUDA 12.4 开发版
FROM nvidia/cuda:12.4.0-devel-ubuntu22.04

# 设置环境变量，避免交互式安装卡住
ENV DEBIAN_FRONTEND=noninteractive

# 安装 Python 和基础依赖
RUN apt-get update && apt-get install -y \\
    python3 \\
    python3-pip \\
    python3-dev \\
    && rm -rf /var/lib/apt/lists/*

# 升级 pip
RUN pip3 install --upgrade pip

# 安装 SGLang 和 OpenAI SDK
RUN pip3 install "sglang[all]" "openai>=1.0.0"

# 暴露服务端口
EXPOSE 30000

# 设置入口命令
ENTRYPOINT ["python3", "-m", "sglang.launch_server"]

# 默认启动参数（可在 docker run 时覆盖）
CMD ["--model-path", "Qwen/Qwen2.5-0.5B-Instruct", "--host", "0.0.0.0", "--port", "30000"]
"""  # Dockerfile 完整内容

print("=" * 60)  # 打印分隔线
print("方式二：自定义 Dockerfile")  # 打印标题
print("=" * 60)  # 打印分隔线
print()  # 打印空行
print(dockerfile_content)  # 输出 Dockerfile 内容

# 将 Dockerfile 保存到临时目录
tmp_dir = tempfile.mkdtemp(prefix="sglang_docker_")  # 创建临时目录
dockerfile_path = os.path.join(tmp_dir, "Dockerfile")  # 拼接文件路径
with open(dockerfile_path, "w") as f:  # 打开文件写入
    f.write(dockerfile_content)  # 写入 Dockerfile 内容
print(f"Dockerfile 已保存到: {dockerfile_path}")  # 提示保存位置

# 打印构建和运行命令
print()  # 打印空行
print("构建镜像命令：")  # 打印构建命令标题
print(f"  docker build -t sglang-custom:latest {tmp_dir}")  # 输出构建命令
print()  # 打印空行
print("运行容器命令：")  # 打印运行命令标题
print("  docker run --gpus all -p 30000:30000 --shm-size=16g \\")  # 输出运行命令第一行
print("    -v ~/.cache/huggingface:/root/.cache/huggingface \\")  # 输出卷挂载参数
print("    sglang-custom:latest")  # 输出镜像名称

## 方式三：Docker Compose

Docker Compose 适合需要多服务协同工作的场景（如搭配反向代理、监控等），也让单服务的管理更加优雅。

**优势：**
- 一个 `docker-compose.yml` 文件描述整个部署拓扑
- `docker compose up -d` 一键启动所有服务
- `docker compose logs` 统一查看日志
- 便于版本管理和团队协作

In [ ]:
# === 方式三：生成 docker-compose.yml ===
import os  # 导入操作系统模块
import tempfile  # 导入临时文件模块

# 定义 docker-compose.yml 内容
compose_content = """version: '3.8'

services:
  sglang:
    image: lmsysorg/sglang:latest
    command: >-
      python -m sglang.launch_server
      --model-path Qwen/Qwen2.5-0.5B-Instruct
      --host 0.0.0.0
      --port 30000
    ports:
      - "30000:30000"
    volumes:
      - ~/.cache/huggingface:/root/.cache/huggingface
    shm_size: '16g'
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    restart: unless-stopped
"""  # Docker Compose 文件完整内容

print("=" * 60)  # 打印分隔线
print("方式三：Docker Compose 配置")  # 打印标题
print("=" * 60)  # 打印分隔线
print()  # 打印空行
print(compose_content)  # 输出 docker-compose.yml 内容

# 将 docker-compose.yml 保存到临时目录
compose_dir = tempfile.mkdtemp(prefix="sglang_compose_")  # 创建临时目录
compose_path = os.path.join(compose_dir, "docker-compose.yml")  # 拼接文件路径
with open(compose_path, "w") as f:  # 打开文件写入
    f.write(compose_content)  # 写入 Compose 文件内容
print(f"docker-compose.yml 已保存到: {compose_path}")  # 提示保存位置

# 打印常用命令
print()  # 打印空行
print("常用 Compose 命令：")  # 打印命令标题
print(f"  cd {compose_dir}")  # 进入目录命令
print("  docker compose up -d          # 后台启动服务")  # 启动命令
print("  docker compose logs -f         # 查看实时日志")  # 日志命令
print("  docker compose down            # 停止并移除容器")  # 停止命令
print("  docker compose restart         # 重启服务")  # 重启命令

## 容器启动后：验证服务

无论使用哪种方式启动容器，都可以通过以下代码验证服务是否正常运行。

> **注意：** 容器启动后需要一定时间加载模型，请等待模型加载完成再执行验证。

In [ ]:
# === 容器启动后：验证服务 ===
import requests  # 导入 HTTP 请求模块
import time  # 导入时间模块
from openai import OpenAI  # 从 openai 包导入 OpenAI 客户端

BASE_URL = "http://127.0.0.1:30000"  # 定义服务基础 URL

# 步骤 1：检查模型列表接口
print("=" * 50)  # 打印分隔线
print("步骤 1: 检查模型列表")  # 打印步骤标题
print("=" * 50)  # 打印分隔线
try:
    resp = requests.get(f"{BASE_URL}/v1/models", timeout=10)  # 发送 GET 请求获取模型列表
    resp.raise_for_status()  # 如果状态码不是 200，抛出异常
    models = resp.json()  # 解析 JSON 响应
    print(f"服务正常! 可用模型: {[m['id'] for m in models['data']]}")  # 输出可用模型
except requests.ConnectionError:  # 如果连接失败
    print("连接失败: 请确认容器已启动且端口映射正确")  # 提示检查容器
except Exception as e:  # 捕获其他异常
    print(f"检查失败: {e}")  # 输出错误信息

# 步骤 2：发送测试聊天请求
print()  # 打印空行
print("=" * 50)  # 打印分隔线
print("步骤 2: 发送测试聊天请求")  # 打印步骤标题
print("=" * 50)  # 打印分隔线
try:
    client = OpenAI(  # 创建 OpenAI 客户端
        base_url=f"{BASE_URL}/v1",  # 设置 API 基础 URL
        api_key="EMPTY"  # SGLang 不需要真实 API Key
    )
    start_time = time.time()  # 记录请求开始时间
    response = client.chat.completions.create(  # 发送聊天补全请求
        model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型
        messages=[{"role": "user", "content": "你好，请用一句话介绍Docker"}],  # 设置对话消息
        max_tokens=100  # 限制最大生成 token 数
    )
    elapsed = time.time() - start_time  # 计算请求耗时
    print(f"模型回复: {response.choices[0].message.content}")  # 输出模型回复
    print(f"请求耗时: {elapsed:.2f} 秒")  # 输出耗时
    print(f"Token 用量: 输入={response.usage.prompt_tokens}, 输出={response.usage.completion_tokens}")  # 输出 token 统计
    print("\n✓ Docker 容器部署验证成功!")  # 输出成功提示
except Exception as e:  # 捕获异常
    print(f"测试失败: {e}")  # 输出错误信息
    print("请确认容器已启动且模型加载完成")  # 提示排查

## 容器管理命令

以下是部署后常用的容器管理命令，方便日常运维。

In [ ]:
# === 容器管理命令速查 ===

# 定义常用 Docker 管理命令
management_commands = {  # 管理命令字典
    "查看运行中的容器": "docker ps",  # 列出所有运行的容器
    "查看所有容器（含已停止）": "docker ps -a",  # 列出所有容器
    "查看容器日志（实时跟踪）": "docker logs -f <容器ID>",  # 实时查看日志
    "查看最近100行日志": "docker logs --tail 100 <容器ID>",  # 查看最近日志
    "停止容器": "docker stop <容器ID>",  # 停止指定容器
    "重启容器": "docker restart <容器ID>",  # 重启指定容器
    "进入容器内部": "docker exec -it <容器ID> /bin/bash",  # 以交互模式进入容器
    "查看容器资源使用": "docker stats <容器ID>",  # 查看 CPU/内存/网络使用
    "删除已停止的容器": "docker rm <容器ID>",  # 删除容器
    "清理悬空镜像": "docker image prune -f",  # 清理无用镜像
}  # 命令字典结束

print("=" * 60)  # 打印分隔线
print("Docker 容器管理命令速查表")  # 打印标题
print("=" * 60)  # 打印分隔线

for desc, cmd in management_commands.items():  # 遍历所有管理命令
    print(f"\n【{desc}】")  # 打印命令描述
    print(f"  $ {cmd}")  # 打印具体命令

# 打印按名称查找容器的提示
print()  # 打印空行
print("=" * 60)  # 打印分隔线
print("提示: 可用 docker ps --filter 'ancestor=lmsysorg/sglang:latest' 查找 SGLang 容器")  # 提示过滤命令

In [ ]:
# === 清理资源 ===
import subprocess  # 导入子进程模块
import tempfile  # 导入临时文件模块
import shutil  # 导入文件操作模块
import glob  # 导入文件匹配模块

print("正在清理临时文件...")  # 打印清理提示

# 清理临时生成的 Dockerfile 和 Compose 文件
tmp_base = tempfile.gettempdir()  # 获取系统临时目录
for pattern in ["sglang_docker_*", "sglang_compose_*"]:  # 遍历要清理的模式
    for d in glob.glob(os.path.join(tmp_base, pattern)):  # 匹配临时目录
        try:
            shutil.rmtree(d)  # 递归删除临时目录
            print(f"  已删除: {d}")  # 输出删除确认
        except Exception as e:  # 捕获异常
            print(f"  删除失败: {d} ({e})")  # 输出错误

print()  # 打印空行
print("提示: 如需停止 Docker 容器中的 SGLang 服务，请在终端运行:")  # 打印停止提示
print("  docker stop $(docker ps -q --filter 'ancestor=lmsysorg/sglang:latest')")  # 输出停止命令
print("清理完成")  # 打印完成提示

## 常见问题排查

### 问题 1：docker: Error response from daemon: could not select device driver "nvidia"

**现象：** 运行 `docker run --gpus all ...` 时报错，提示找不到 nvidia 设备驱动。

**可能原因：**
- 未安装 NVIDIA Container Toolkit
- 安装后未重启 Docker 服务

**解决：**
```bash
# 安装 NVIDIA Container Toolkit
sudo apt-get install -y nvidia-container-toolkit
# 配置 Docker 运行时
sudo nvidia-ctk runtime configure --runtime=docker
# 重启 Docker
sudo systemctl restart docker
```

### 问题 2：容器被 OOM Killed（内存不足被杀）

**现象：** 容器启动一段时间后自动退出，`docker inspect` 显示 OOMKilled: true。

**可能原因：**
- 共享内存（/dev/shm）不足，默认仅 64MB
- 模型过大，GPU 显存不够

**解决：**
```bash
# 增大共享内存
docker run --gpus all --shm-size=16g ...

# 或换用更小的模型
docker run --gpus all --shm-size=16g \
  lmsysorg/sglang:latest \
  python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-0.5B-Instruct \
  --host 0.0.0.0 --port 30000
```

---

### 结语

Docker 容器化部署让 SGLang 推理服务的交付变得标准化和可复现。本 Notebook 介绍了三种方式：官方镜像一键部署、自定义 Dockerfile 灵活定制、Docker Compose 编排管理。根据实际需求选择合适的方式，即可在生产环境中快速上线 LLM 推理服务。